In [2]:
import sys
sys.path.append("..")
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from datasets import load_from_disk
from tqdm import tqdm

from src.data import PROJECT_ROOT
from src.llm_upgrade import wrap_for_transformer_lens
from src.sae import TopKSAE
from src.interventions import (
    measure_intervention_effect2,
    rank_latents_by_causal_effect,
    extract_source_activation,
    create_ablation_hook,
    create_patching_hook
)

In [3]:
EXP_ID = "exp8-2"
MODEL_NAME = "EleutherAI/pythia-1b-deduped"
VARIANT = "depth-1"
ADAPTER_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-1000")
PROBING_JSON = PROJECT_ROOT / "results/probing/probe_1b(seq_qlora)_depth-1_resid_post_last.json"
with open(PROBING_JSON, "r", encoding="utf-8") as f:
    probing_data = json.load(f)
BEST_LAYER = probing_data["summary"]["best_layer"]
SAE_DIR = PROJECT_ROOT / f"results/sae/{EXP_ID}/layer_{BEST_LAYER}"
SAE_PATH = str(SAE_DIR / "sae_final.pt")

In [4]:
BASELINE_DIR = PROJECT_ROOT / "results/baselines" / EXP_ID
BASELINE_CACHE_DIR = BASELINE_DIR / "cache"

In [5]:
N_CAUSAL_SAMPLES = 500
BATCH_SIZE = 64
TOP_K_LATENTS = 20

In [6]:
torch.cuda.empty_cache()

# Загрузка модели, датасета, SAE

In [7]:
hooked_model, tokenizer = wrap_for_transformer_lens(MODEL_NAME, ADAPTER_PATH, device="cuda")
hooked_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-15): 16 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
    

In [8]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
full_test = load_from_disk(str(cache_path))["test"]

In [9]:
checkpoint = torch.load(SAE_PATH, map_location="cuda", weights_only=False)
K_SAE_CFG = checkpoint['config']['k']
D_SAE = checkpoint['config']['d_sae']
D_MODEL = checkpoint['config']['d_in']

In [10]:
sae = TopKSAE(d_in=D_MODEL, d_sae=D_SAE, k=K_SAE_CFG).to("cuda")
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

TopKSAE()

In [11]:
activations = np.load(SAE_DIR / "test_activations_formatted.npy")
labels = np.load(SAE_DIR / "test_labels_formatted.npy")
llm_preds = torch.load(BASELINE_CACHE_DIR / "test_acts_preds.pt", map_location="cpu", weights_only=False)["preds"].numpy()

In [12]:
eval_prefix = "{theory} {assertion} The assertion is"
def format_text(item):
    if "theory" in item and "assertion" in item:
        return eval_prefix.format(theory=item["theory"], assertion=item["assertion"])
    return item.get("text", "")

dataset_texts = [format_text(full_test[i]) for i in range(len(labels))]

In [13]:
rng = np.random.default_rng(42)

idx_true = np.where(labels == 1)[0]
idx_false = np.where(labels == 0)[0]
selected_idx = np.concatenate([
    rng.choice(idx_true, size=min(int(N_CAUSAL_SAMPLES/2), len(idx_true)), replace=False),
    rng.choice(idx_false, size=min(int(N_CAUSAL_SAMPLES/2), len(idx_false)), replace=False)
])
rng.shuffle(selected_idx)

In [14]:
texts_causal = [dataset_texts[i] for i in selected_idx]
llm_causal = llm_preds[selected_idx]
labels_causal = labels[selected_idx]
acts_causal = activations[selected_idx]

In [15]:
print(f"Выборка для каузального анализа: {len(texts_causal)} примеров (баланс: {np.mean(labels_causal):.2f})")

Выборка для каузального анализа: 500 примеров (баланс: 0.50)


In [16]:
true_ids = tokenizer.encode("True", add_special_tokens=False)
false_ids = tokenizer.encode("False", add_special_tokens=False)
TRUE_ID = true_ids[-1] if true_ids else 0
FALSE_ID = false_ids[-1] if false_ids else 0

In [17]:
HOOK_NAME = f"blocks.{BEST_LAYER}.hook_resid_post"

# 1. Абляция: ранжирование латентов по каузальному эффекту

In [21]:
def create_ablation_hook2(sae, latent_idx, device="cuda"):
    """Корректный хук для TransformerLens"""
    def hook(tensor, hook):
        tensor = tensor.to(torch.float32)
        last_vec = tensor[:, -1, :].unsqueeze(1)  # [batch, 1, d_model]
        sparse, _ = sae(last_vec)
        sparse[:, :, latent_idx] = 0.0
        # Декодирование без b_dec
        recon = sparse @ sae.W_dec
        tensor[:, -1, :] = recon.squeeze(1)
        return tensor
    return hook

In [ ]:
def measure_intervention_effect2(hooked_model, sae, texts, llm_preds, labels_gt, prompt_lens,
                                 latent_idx, hook_name, true_id, false_id, batch_size=32, device="cuda"):
    n = len(texts)
    pred_base = []

    # Базовые предсказания
    with torch.no_grad():
        for i in range(0, n, batch_size):
            tokens = hooked_model.to_tokens(texts[i:i+batch_size], prepend_bos=True).to(device)
            logits = hooked_model(tokens)
            for b in range(tokens.shape[0]):
                seq_len = prompt_lens[i+b]
                safe_len = min(seq_len, logits.shape[1] - 1)
                pred_base.append(1 if logits[b, safe_len, true_id] > logits[b, safe_len, false_id] else 0)

    # Предсказания после абляции
    hook = create_ablation_hook2(sae, latent_idx, device)
    pred_interv = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            tokens = hooked_model.to_tokens(texts[i:i+batch_size], prepend_bos=True).to(device)
            with hooked_model.hooks(fwd_hooks=[(hook_name, hook)]):
                logits = hooked_model(tokens)
            for b in range(tokens.shape[0]):
                seq_len = prompt_lens[i+b]
                safe_len = min(seq_len, logits.shape[1] - 1)
                pred_interv.append(1 if logits[b, safe_len, true_id] > logits[b, safe_len, false_id] else 0)

    pred_base = np.array(pred_base)
    pred_interv = np.array(pred_interv)

    return {
        "latent_idx": latent_idx,
        "delta_fidelity": round(float(np.mean(pred_base == llm_preds[:n])) - float(np.mean(pred_interv == llm_preds[:n])), 4),
        "delta_accuracy": round(float(np.mean(pred_base == labels_gt[:n])) - float(np.mean(pred_interv == labels_gt[:n])), 4),
        "flip_rate": round(float(np.mean(pred_base != pred_interv)), 4)
    }

In [23]:
prompt_lens = [len(tokenizer.encode(txt, add_special_tokens=False)) for txt in texts_causal]

In [24]:
from src.rule_extraction import rank_features_by_logic
top_feat, corr_label, corr_logic, combined = rank_features_by_logic(acts_causal, llm_causal, texts_causal)
candidate_latents = top_feat[:TOP_K_LATENTS].tolist()

In [25]:
ranking_sae = []
for latent_idx in tqdm(candidate_latents, desc="Causal ablation (SAE)"):
    metrics = measure_intervention_effect2(
        hooked_model, sae, texts_causal, llm_causal, labels_causal, prompt_lens,
        latent_idx, HOOK_NAME, TRUE_ID, FALSE_ID, batch_size=BATCH_SIZE, device="cuda"
    )
    ranking_sae.append(metrics)

Causal ablation (SAE): 100%|██████████| 20/20 [16:06<00:00, 48.32s/it]


In [26]:
df_causal = pd.DataFrame(ranking_sae).sort_values("delta_fidelity", ascending=False).reset_index(drop=True)
df_causal.to_csv(BASELINE_DIR / "causal_ablation_results_v2.csv", index=False)

In [29]:
df_causal = pd.read_csv(BASELINE_DIR / "causal_ablation_results_v2.csv")
display(df_causal)

,latent_idx,delta_fidelity,delta_accuracy,flip_rate
0,1097,0.006,0.006,0.010
1,2134,0.006,0.006,0.010
2,4549,0.006,0.006,0.010
3,1931,0.006,0.006,0.010
4,3255,0.006,0.006,0.010
5,4912,0.006,0.006,0.010
6,6669,0.006,0.006,0.010
7,3644,0.006,0.006,0.010
8,7895,0.006,0.006,0.010
9,5327,0.006,0.006,0.010


In [20]:
ranking_sae = []

for latent_idx in tqdm(candidate_latents, desc="Causal ablation (SAE)"):
    metrics = measure_intervention_effect2(
        hooked_model, sae, tokenizer, texts_causal, llm_causal, labels_causal,
        latent_idx, HOOK_NAME, TRUE_ID, FALSE_ID, batch_size=BATCH_SIZE, device="cuda"
    )
    ranking_sae.append(metrics)

Causal ablation (SAE):   0%|          | 0/20 [00:14<?, ?it/s]


TypeError: create_ablation_hook2.<locals>.hook() got an unexpected keyword argument 'hook'

In [ ]:


for latent_idx in tqdm(candidate_latents, desc="Causal ablation (SAE)"):
    metrics = measure_intervention_effect(
        hooked_model, sae, tokenizer, texts_causal, llm_causal, labels_causal,
        latent_idx, HOOK_NAME, TRUE_ID, FALSE_ID, batch_size=BATCH_SIZE, device="cuda"
    )
    ranking_sae.append(metrics)

Causal ablation (SAE):   0%|          | 0/20 [00:00<?, ?it/s]


TypeError: measure_intervention_effect() missing 1 required positional argument: 'false_id'

In [ ]:
n = len(texts_causal)
fidelity_before, accuracy_before = 0, 0
fidelity_after, accuracy_after = 0, 0
flips = 0

# Предсказания модели до интервенции
pred_base = []
with torch.no_grad():
    for i in range(0, n, BATCH_SIZE):
        tokens = hooked_model.to_tokens(texts_causal[i:i+BATCH_SIZE], prepend_bos=True).to(device)
        logits = hooked_model(tokens)
        for b in range(logits.shape[0]):
            l_true = logits[b, -1, TRUE_ID]
            l_false = logits[b, -1, FALSE_ID]
            pred_base.append(1 if l_true > l_false else 0)
pred_base = np.array(pred_base)

fidelity_before = float(np.mean(pred_base == llm_preds[:n]))
accuracy_before = float(np.mean(pred_base == labels_gt[:n]))

# Предсказания после абляции
hook = create_ablation_hook(sae, latent_idx, HOOK_NAME, device)
pred_interv = []
with torch.no_grad():
    for i in range(0, n, BATCH_SIZE):
        tokens = hooked_model.to_tokens(texts_causal[i:i+BATCH_SIZE], prepend_bos=True).to(device)
        logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook)])
        for b in range(logits.shape[0]):
            l_true = logits[b, -1, TRUE_ID]
            l_false = logits[b, -1, FALSE_ID]
            pred_interv.append(1 if l_true > l_false else 0)
pred_interv = np.array(pred_interv)

fidelity_after = float(np.mean(pred_interv == llm_preds[:n]))
accuracy_after = float(np.mean(pred_interv == labels_gt[:n]))
flips = float(np.mean(pred_base != pred_interv))

return {
    "latent_idx": latent_idx,
    "fidelity_before": round(fidelity_before, 4),
    "fidelity_after": round(fidelity_after, 4),
    "delta_fidelity": round(fidelity_before - fidelity_after, 4),
    "accuracy_before": round(accuracy_before, 4),
    "accuracy_after": round(accuracy_after, 4),
    "delta_accuracy": round(accuracy_before - accuracy_after, 4),
    "flip_rate": round(flips, 4)
}

In [21]:
preds_base = []
prompt_lens = []

In [22]:
for txt in texts_causal:
    p_len = len(tokenizer.encode(txt, add_special_tokens=False))
    prompt_lens.append(p_len)

In [ ]:
with torch.no_grad():
    for i in range(0, len(texts_causal), BATCH_SIZE):
        batch_texts = texts_causal[i:i+BATCH_SIZE]
        batch_lens = prompt_lens[i:i+BATCH_SIZE]
        tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
        logits = hooked_model(tokens)

        # Индексация по длине промпта
        for j, seq_len in enumerate(batch_lens):
            l_true = logits[j, seq_len, TRUE_ID]
            l_false = logits[j, seq_len, FALSE_ID]
            preds_base.append(1 if l_true > l_false else 0)
preds_base = np.array(preds_base)

In [24]:
baseline_alignment = float(np.mean(preds_base == llm_causal))
baseline_alignment

0.996

In [25]:
from src.rule_extraction import rank_features_by_logic
top_feat, corr_label, corr_logic, combined = rank_features_by_logic(acts_causal, llm_causal, texts_causal)
candidate_latents = top_feat[:TOP_K_LATENTS].tolist()

In [ ]:
# ranking = []

# for latent_idx in tqdm(candidate_latents, desc="Causal ablation"):
#     hook = create_ablation_hook(latent_idx, sae, HOOK_NAME, ablation_value=0.0, device="cuda")
#     preds_interv = []

#     with torch.no_grad():
#         for i in range(0, len(texts_causal), BATCH_SIZE):
#             batch_texts = texts_causal[i:i+BATCH_SIZE]
#             batch_lens = prompt_lens[i:i+BATCH_SIZE]
#             tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
#             logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook)])

#             for j, seq_len in enumerate(batch_lens):
#                 l_true = logits[j, seq_len, TRUE_ID]
#                 l_false = logits[j, seq_len, FALSE_ID]
#                 preds_interv.append(1 if l_true > l_false else 0)
#     preds_interv = np.array(preds_interv)

#     fid_before = float(np.mean(preds_base == llm_causal))
#     fid_after = float(np.mean(preds_interv == llm_causal))
#     acc_before = float(np.mean(preds_base == labels_causal))
#     acc_after = float(np.mean(preds_interv == labels_causal))
#     flip_rate = float(np.mean(preds_base != preds_interv))

#     ranking.append({
#         "latent_idx": latent_idx,
#         "fidelity_before": round(fid_before, 4),
#         "fidelity_after": round(fid_after, 4),
#         "delta_fidelity": round(fid_before - fid_after, 4),
#         "accuracy_before": round(acc_before, 4),
#         "accuracy_after": round(acc_after, 4),
#         "delta_accuracy": round(acc_before - acc_after, 4),
#         "flip_rate": round(flip_rate, 4)
#     })

Causal ablation: 100%|██████████| 20/20 [06:05<00:00, 18.30s/it]


In [ ]:
# ranking.sort(key=lambda x: x["delta_fidelity"], reverse=True)
# df_causal = pd.DataFrame(ranking)
# df_causal.to_csv(BASELINE_DIR / "causal_ablation_results.csv", index=False)

In [33]:
df_causal = pd.read_csv(BASELINE_DIR / "causal_ablation_results.csv")

In [34]:
df_causal

,latent_idx,fidelity_before,fidelity_after,delta_fidelity,accuracy_before,accuracy_after,delta_accuracy,flip_rate
0,1097,0.996,0.990,0.006,0.814,0.808,0.006,0.010
1,2134,0.996,0.990,0.006,0.814,0.808,0.006,0.010
2,6102,0.996,0.990,0.006,0.814,0.808,0.006,0.010
3,7090,0.996,0.990,0.006,0.814,0.808,0.006,0.010
4,54,0.996,0.990,0.006,0.814,0.808,0.006,0.010
5,7532,0.996,0.990,0.006,0.814,0.808,0.006,0.010
6,4599,0.996,0.990,0.006,0.814,0.808,0.006,0.010
7,7849,0.996,0.990,0.006,0.814,0.808,0.006,0.010
8,5575,0.996,0.990,0.006,0.814,0.808,0.006,0.010
9,5327,0.996,0.990,0.006,0.814,0.808,0.006,0.010


In [39]:
ranking_sae = []

for latent_idx in tqdm(candidate_latents, desc="Causal ablation (SAE)"):
    # Используем SAE-хук: encode -> обнуление -> decode -> замена
    hook = create_ablation_hook(latent_idx, sae, HOOK_NAME, ablation_value=0.0, device="cuda")
    preds_interv = []

    with torch.no_grad():
        for i in range(0, len(texts_causal), BATCH_SIZE):
            batch_texts = texts_causal[i:i+BATCH_SIZE]
            batch_lens = prompt_lens[i:i+BATCH_SIZE]
            tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
            logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook)])

            for j, seq_len in enumerate(batch_lens):
                l_true = logits[j, seq_len, TRUE_ID]
                l_false = logits[j, seq_len, FALSE_ID]
                preds_interv.append(1 if l_true > l_false else 0)
    preds_interv = np.array(preds_interv)

    fid_before = float(np.mean(preds_base == llm_causal))
    fid_after = float(np.mean(preds_interv == llm_causal))
    acc_before = float(np.mean(preds_base == labels_causal))
    acc_after = float(np.mean(preds_interv == labels_causal))
    flip_rate = float(np.mean(preds_base != preds_interv))

    ranking_sae.append({
        "latent_idx": latent_idx,
        "fidelity_before": round(fid_before, 4),
        "fidelity_after": round(fid_after, 4),
        "delta_fidelity": round(fid_before - fid_after, 4),
        "accuracy_before": round(acc_before, 4),
        "accuracy_after": round(acc_after, 4),
        "delta_accuracy": round(acc_before - acc_after, 4),
        "flip_rate": round(flip_rate, 4)
    })

Causal ablation (SAE): 100%|██████████| 20/20 [09:52<00:00, 29.61s/it]


In [40]:
ranking_sae.sort(key=lambda x: x["delta_fidelity"], reverse=True)
df_causal = pd.DataFrame(ranking_sae)
df_causal.to_csv(BASELINE_DIR / "causal_ablation(SAE)_results.csv", index=False)

In [41]:
df_causal = pd.read_csv(BASELINE_DIR / "causal_ablation(SAE)_results.csv")

In [42]:
df_causal

,latent_idx,fidelity_before,fidelity_after,delta_fidelity,accuracy_before,accuracy_after,delta_accuracy,flip_rate
0,1097,0.996,0.990,0.006,0.814,0.808,0.006,0.010
1,2134,0.996,0.990,0.006,0.814,0.808,0.006,0.010
2,6102,0.996,0.990,0.006,0.814,0.808,0.006,0.010
3,7090,0.996,0.990,0.006,0.814,0.808,0.006,0.010
4,54,0.996,0.990,0.006,0.814,0.808,0.006,0.010
5,7532,0.996,0.990,0.006,0.814,0.808,0.006,0.010
6,4599,0.996,0.990,0.006,0.814,0.808,0.006,0.010
7,7849,0.996,0.990,0.006,0.814,0.808,0.006,0.010
8,5575,0.996,0.990,0.006,0.814,0.808,0.006,0.010
9,5327,0.996,0.990,0.006,0.814,0.808,0.006,0.010


In [60]:
with torch.no_grad():
    tokens = hooked_model.to_tokens([texts_causal[0]], prepend_bos=True).to("cuda")

    _, cache_orig = hooked_model.run_with_cache(tokens, names_filter=lambda n: n == HOOK_NAME)
    act_orig = cache_orig[HOOK_NAME][0, -1, :].to(torch.float32).cpu().numpy()

    hook = create_ablation_hook(1097, sae, HOOK_NAME, device="cuda")
    _, cache_abl = hooked_model.run_with_cache(tokens, fwd_hooks=[(HOOK_NAME, hook)])
    act_abl = cache_abl[HOOK_NAME][0, -1, :].to(torch.float32).cpu().numpy()

    print(f"||Δactivation|| = {np.linalg.norm(act_abl - act_orig):.4f}")

TypeError: HookedTransformer.forward() got an unexpected keyword argument 'fwd_hooks'

In [61]:
def check_ablation_specificity(hooked_model, sae, tokenizer, text, latent_idx, hook_name, device="cuda"):
    tokens = hooked_model.to_tokens([text], prepend_bos=True).to(device)

    with torch.no_grad():
        _, cache = hooked_model.run_with_cache(tokens, names_filter=lambda n: n == hook_name)
        act_orig = cache[hook_name][0, -1, :].unsqueeze(0).to(torch.float32)
        sparse_orig, _ = sae(act_orig)

        hook = create_ablation_hook(latent_idx, sae, hook_name, device=device)
        _, cache_abl = hooked_model.run_with_cache(tokens, fwd_hooks=[(hook_name, hook)])
        act_abl = cache_abl[hook_name][0, -1, :].unsqueeze(0).to(torch.float32)
        sparse_abl, _ = sae(act_abl)

        act_change = torch.norm(act_abl - act_orig).item()
        latent_change = torch.norm(sparse_abl - sparse_orig).item()
        target_zeroed = sparse_abl[0, 0, latent_idx].item()

        return {
            "||Δactivation||": act_change,
            "||Δsparse||": latent_change,
            "target_latent_zeroed": target_zeroed,
            "orig_target_value": sparse_orig[0, 0, latent_idx].item()
        }

result = check_ablation_specificity(hooked_model, sae, tokenizer, texts_causal[0], 1097, HOOK_NAME, device="cuda")
print(result)

TypeError: HookedTransformer.forward() got an unexpected keyword argument 'fwd_hooks'

In [47]:
# ---------- Холостой проход SAE (baseline pass-through) ----------
def create_pass_through_hook(sae, hook_name, device="cuda"):
    def hook(act, hook):
        act = act.to(torch.float32)
        last_vec = act[:, -1, :].unsqueeze(1)  # [batch, 1, d_model]
        sparse, _ = sae(last_vec)
        reconstructed = sae.decode(sparse)
        act[:, -1, :] = reconstructed.squeeze(1)
        return act
    return hook

pass_hook = create_pass_through_hook(sae, HOOK_NAME, "cuda")
preds_sae_base = []
with torch.no_grad():
    for i in range(0, len(texts_causal), BATCH_SIZE):
        batch_texts = texts_causal[i:i+BATCH_SIZE]
        batch_lens = prompt_lens[i:i+BATCH_SIZE]
        tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
        logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, pass_hook)])
        for j, seq_len in enumerate(batch_lens):
            l_true = logits[j, seq_len, TRUE_ID]
            l_false = logits[j, seq_len, FALSE_ID]
            preds_sae_base.append(1 if l_true > l_false else 0)
preds_sae_base = np.array(preds_sae_base)

fidelity_base_sae = float(np.mean(preds_sae_base == llm_causal))
acc_base_sae = float(np.mean(preds_sae_base == labels_causal))
print(f"Fidelity после SAE (без абляции): {fidelity_base_sae:.4f}")
print(f"Accuracy после SAE (без абляции): {acc_base_sae:.4f}")

Fidelity после SAE (без абляции): 0.9900
Accuracy после SAE (без абляции): 0.8080


In [49]:
ranking_sae = []
for latent_idx in tqdm(candidate_latents, desc="Causal ablation (SAE)"):
    hook = create_ablation_hook(latent_idx, sae, HOOK_NAME, ablation_value=0.0, device="cuda")
    preds_interv = []
    with torch.no_grad():
        for i in range(0, len(texts_causal), BATCH_SIZE):
            batch_texts = texts_causal[i:i+BATCH_SIZE]
            batch_lens = prompt_lens[i:i+BATCH_SIZE]
            tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
            logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook)])
            for j, seq_len in enumerate(batch_lens):
                l_true = logits[j, seq_len, TRUE_ID]
                l_false = logits[j, seq_len, FALSE_ID]
                preds_interv.append(1 if l_true > l_false else 0)
    preds_interv = np.array(preds_interv)

    fid_after = float(np.mean(preds_interv == llm_causal))
    delta_fid = fidelity_base_sae - fid_after
    acc_after = float(np.mean(preds_interv == labels_causal))
    delta_acc = acc_base_sae - acc_after
    flip_rate = float(np.mean(preds_sae_base != preds_interv))  # относительно SAE‑baseline

    ranking_sae.append({
        "latent_idx": latent_idx,
        "fidelity_base_sae": round(fidelity_base_sae, 4),
        "fidelity_after": round(fid_after, 4),
        "delta_fidelity": round(delta_fid, 4),
        "accuracy_base_sae": round(acc_base_sae, 4),
        "accuracy_after": round(acc_after, 4),
        "delta_accuracy": round(delta_acc, 4),
        "flip_rate": round(flip_rate, 4)
    })

Causal ablation (SAE): 100%|██████████| 20/20 [09:52<00:00, 29.60s/it]


In [50]:
ranking_sae.sort(key=lambda x: x["delta_fidelity"], reverse=True)
df_causal = pd.DataFrame(ranking_sae)
df_causal.to_csv(BASELINE_DIR / "causal_ablation_results2.csv", index=False)

In [65]:
df_causal = pd.read_csv(BASELINE_DIR / "causal_ablation_results2.csv")
df_causal.head(10)

,latent_idx,fidelity_base_sae,fidelity_after,delta_fidelity,accuracy_base_sae,accuracy_after,delta_accuracy,flip_rate
0,1097,0.99,0.99,0.0,0.808,0.808,0.0,0.0
1,2134,0.99,0.99,0.0,0.808,0.808,0.0,0.0
2,6102,0.99,0.99,0.0,0.808,0.808,0.0,0.0
3,7090,0.99,0.99,0.0,0.808,0.808,0.0,0.0
4,54,0.99,0.99,0.0,0.808,0.808,0.0,0.0
5,7532,0.99,0.99,0.0,0.808,0.808,0.0,0.0
6,4599,0.99,0.99,0.0,0.808,0.808,0.0,0.0
7,7849,0.99,0.99,0.0,0.808,0.808,0.0,0.0
8,5575,0.99,0.99,0.0,0.808,0.808,0.0,0.0
9,5327,0.99,0.99,0.0,0.808,0.808,0.0,0.0


In [53]:
all_latent_indices = list(range(D_SAE))
random_pool = [idx for idx in all_latent_indices if idx not in candidate_latents]
n_random = 20
random_deltas = []
for _ in tqdm(range(n_random), desc="Random ablation"):
    rnd_idx = np.random.choice(random_pool)
    hook = create_ablation_hook(rnd_idx, sae, HOOK_NAME, ablation_value=0.0, device="cuda")
    preds_rnd = []
    with torch.no_grad():
        for i in range(0, len(texts_causal), BATCH_SIZE):
            batch_texts = texts_causal[i:i+BATCH_SIZE]
            batch_lens = prompt_lens[i:i+BATCH_SIZE]
            tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
            logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook)])
            for j, seq_len in enumerate(batch_lens):
                l_true = logits[j, seq_len, TRUE_ID]
                l_false = logits[j, seq_len, FALSE_ID]
                preds_rnd.append(1 if l_true > l_false else 0)
    preds_rnd = np.array(preds_rnd)
    fid_rnd = float(np.mean(preds_rnd == llm_causal))
    random_deltas.append(fidelity_base_sae - fid_rnd)

random_deltas = np.array(random_deltas)

Random ablation: 100%|██████████| 20/20 [09:55<00:00, 29.79s/it]


In [54]:
random_deltas

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [57]:
from itertools import combinations

def ablate_group(latent_indices, texts, prompt_lens, true_id, false_id, batch_size=BATCH_SIZE):
    hooks = [create_ablation_hook(idx, sae, HOOK_NAME, ablation_value=0.0, device="cuda") for idx in latent_indices]
    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_lens = prompt_lens[i:i+batch_size]
            tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
            logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook) for hook in hooks])
            for j, seq_len in enumerate(batch_lens):
                l_true = logits[j, seq_len, true_id]
                l_false = logits[j, seq_len, false_id]
                preds.append(1 if l_true > l_false else 0)
    return np.array(preds)

group_sizes = [3, 5, 10]
top_latent_list = df_causal["latent_idx"].tolist()

for size in group_sizes:
    # Топ‑латент группа
    top_group = top_latent_list[:size]
    preds_top = ablate_group(top_group, texts_causal, prompt_lens, TRUE_ID, FALSE_ID)
    delta_top = fidelity_base_sae - np.mean(preds_top == llm_causal)

    # Случайная группа (усреднение по 10 случайным наборам)
    random_deltas = []
    for _ in range(10):
        random_group = np.random.choice(random_pool, size=size, replace=False).tolist()
        preds_rnd = ablate_group(random_group, texts_causal, prompt_lens, TRUE_ID, FALSE_ID)
        random_deltas.append(fidelity_base_sae - np.mean(preds_rnd == llm_causal))
    random_delta_mean = np.mean(random_deltas)
    random_delta_std = np.std(random_deltas)

    print(f"Группа {size}: топ δ = {delta_top:.4f}, случайные δ = {random_delta_mean:.4f} ± {random_delta_std:.4f}")

Группа 3: топ δ = 0.0000, случайные δ = 0.0000 ± 0.0000
Группа 5: топ δ = 0.0020, случайные δ = 0.0000 ± 0.0000
Группа 10: топ δ = 0.0000, случайные δ = 0.0000 ± 0.0000


In [ ]:
def measure_logit_change(latent_indices, texts, prompt_lens, true_id, false_id, batch_size=BATCH_SIZE):
    hooks = [create_ablation_hook(idx, sae, HOOK_NAME, ablation_value=0.0, device="cuda") for idx in latent_indices]
    diff_before = []
    diff_after = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_lens = prompt_lens[i:i+batch_size]
            tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
            logits_base = hooked_model(tokens)
            logits_abl = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook) for hook in hooks])
            for j, seq_len in enumerate(batch_lens):
                diff_before.append(logits_base[j, seq_len, true_id].item() - logits_base[j, seq_len, false_id].item())
                diff_after.append(logits_abl[j, seq_len, true_id].item() - logits_abl[j, seq_len, false_id].item())
    diff_before = np.array(diff_before)
    diff_after = np.array(diff_after)
    return diff_before, diff_after

# Пример для топ-5
top5 = top_latent_list[:5]
diff_before, diff_after = measure_logit_change(top5, texts_causal, prompt_lens, TRUE_ID, FALSE_ID)
mean_shift = np.mean(diff_after - diff_before)

In [59]:
print(f"Средний сдвиг разности логитов True−False для топ-5: {mean_shift:.4f}")

Средний сдвиг разности логитов True−False для топ-5: 0.0088


In [66]:
b_dec_norm = sae.b_dec.norm().item()
# Средняя норма реальной активации
act_mean_norm = np.linalg.norm(activations.mean(axis=0))
print(f"Норма b_dec: {b_dec_norm:.4f}")
print(f"Норма среднего вектора активаций: {act_mean_norm:.4f}")
# Если b_dec_norm >> act_mean_norm, то SAE вырожден.

Норма b_dec: 0.1714
Норма среднего вектора активаций: 52.8171


In [67]:
def create_full_ablation_hook(sae, hook_name, device="cuda"):
    def hook(act, hook):
        act = act.to(torch.float32)
        last_vec = act[:, -1, :].unsqueeze(1)
        sparse, _ = sae(last_vec)
        # Обнуляем все латенты
        sparse = torch.zeros_like(sparse)
        reconstructed = sae.decode(sparse)  # останется только b_dec
        act[:, -1, :] = reconstructed.squeeze(1)
        return act
    return hook

full_hook = create_full_ablation_hook(sae, HOOK_NAME, "cuda")
preds_full = []
with torch.no_grad():
    for i in range(0, len(texts_causal), BATCH_SIZE):
        batch_texts = texts_causal[i:i+BATCH_SIZE]
        batch_lens = prompt_lens[i:i+BATCH_SIZE]
        tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
        logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, full_hook)])
        for j, seq_len in enumerate(batch_lens):
            l_true = logits[j, seq_len, TRUE_ID]
            l_false = logits[j, seq_len, FALSE_ID]
            preds_full.append(1 if l_true > l_false else 0)
preds_full = np.array(preds_full)
fid_full = float(np.mean(preds_full == llm_causal))
print(f"Fidelity после полной абляции SAE (все латенты = 0): {fid_full:.4f}")

Fidelity после полной абляции SAE (все латенты = 0): 0.9880


In [68]:
import torch
acts_tensor = torch.from_numpy(activations[:500]).float().cuda()  # первые 500 примеров
with torch.no_grad():
    sparse, recon_full = sae(acts_tensor)
    recon_zero = sae.decode(torch.zeros_like(sparse))
    diff = (recon_full - recon_zero).norm(dim=1).mean().item()
    print(f"Среднее L2-изменение при обнулении латентов: {diff:.4f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (500x8192 and 2048x8192)

In [69]:
# Вычисляем среднее по train_acts (загрузите train_acts аналогично тому, как в baseline)
train_acts_path = SAE_DIR / "activations.pt"
train_acts_data = torch.load(train_acts_path, map_location="cpu", weights_only=False)
train_acts = train_acts_data["activations"].float()
mean_vec = train_acts.mean(dim=0).numpy()  # [d_model]

def create_centered_ablation_hook(latent_idx, sae, hook_name, mean_vec, ablation_value=0.0, device="cuda"):
    mean_tensor = torch.from_numpy(mean_vec).float().to(device)
    def hook(act, hook):
        act = act.to(torch.float32)
        last_vec = act[:, -1, :].unsqueeze(1)
        centered = last_vec - mean_tensor
        sparse, _ = sae(centered)
        sparse[:, :, latent_idx] = ablation_value
        reconstructed = sae.decode(sparse)
        act[:, -1, :] = (reconstructed + mean_tensor).squeeze(1)
        return act
    return hook

# Тест для топ-1 латента
test_latent = int(df_causal.loc[0, "latent_idx"])
hook_test = create_centered_ablation_hook(test_latent, sae, HOOK_NAME, mean_vec, ablation_value=0.0, device="cuda")
preds_test = []
with torch.no_grad():
    for i in range(0, len(texts_causal), BATCH_SIZE):
        batch_texts = texts_causal[i:i+BATCH_SIZE]
        batch_lens = prompt_lens[i:i+BATCH_SIZE]
        tokens = hooked_model.to_tokens(batch_texts, prepend_bos=True).to("cuda")
        logits = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook_test)])
        for j, seq_len in enumerate(batch_lens):
            l_true = logits[j, seq_len, TRUE_ID]
            l_false = logits[j, seq_len, FALSE_ID]
            preds_test.append(1 if l_true > l_false else 0)
preds_test = np.array(preds_test)
fid_centered = float(np.mean(preds_test == llm_causal))
print(f"Fidelity после центрированной абляции латента #{test_latent}: {fid_centered:.4f}")
print(f"Эффект: {fidelity_base_sae - fid_centered:.4f}")

Fidelity после центрированной абляции латента #1097: 0.9920
Эффект: -0.0020


In [70]:
prompt_lens = [len(tokenizer.encode(txt, add_special_tokens=False)) for txt in texts_causal]

In [71]:
def create_debug_hook(sae, hook_name):
    def hook(tensor, hook_obj):
        tensor = tensor.to(torch.float32)
        orig_norm = torch.norm(tensor[:, -1, :]).item()

        last_vec = tensor[:, -1, :].unsqueeze(1)
        sparse, _ = sae(last_vec)
        sparse_zero = torch.zeros_like(sparse)
        recon = sae.decode(sparse_zero)
        recon_norm = torch.norm(recon).item()

        tensor[:, -1, :] = recon.squeeze(1)
        print(f"  [HOOK] orig_norm={orig_norm:.4f} | recon_norm={recon_norm:.4f} | delta_norm={orig_norm - recon_norm:.4f}")
        return tensor
    return hook

In [72]:
with torch.no_grad():
    tokens = hooked_model.to_tokens(texts_causal[:BATCH_SIZE], prepend_bos=True).to("cuda")
    debug_hook = create_debug_hook(sae, HOOK_NAME)
    _ = hooked_model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, debug_hook)])

TypeError: create_debug_hook.<locals>.hook() got an unexpected keyword argument 'hook'